## COMPARISON CHARTS 
### **1) BAR CHART**

### 📊 Bar Chart Explanation

**Analytical Question:**

**Which countries generated the most electricity in 2025, and how large is the gap between the leading country and the rest of the top producers?**

This chart shows the **Top 10 countries by electricity generation**.

- Countries with longer bars generate more electricity.
- The highlighted bar represents the country with the highest electricity generation.
- A horizontal bar chart was used to make country names easier to read.
- The chart helps compare electricity production levels between countries.

In [1]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from IPython.display import Markdown, display

BASE_DIR = Path.cwd().resolve().parent
csv_path = BASE_DIR / "data" / "owid_energy_visualization_cleaned.csv"

df = pd.read_csv(csv_path)

latest_year = df["year"].max()
latest_df = df[df["year"] == latest_year].copy()

exclude_names = [
    "World", "G20 (Ember)", "Asia (Ember)", "Oecd (Ember)",
    "OECD (Ember)", "G7 (Ember)", "North America (Ember)",
    "Europe (Ember)", "Europe"
]

latest_df = latest_df[~latest_df["country"].isin(exclude_names)]

top10 = (
    latest_df.groupby("country", as_index=False)["electricity_generation"]
    .sum()
    .sort_values("electricity_generation", ascending=False)
    .head(10)
)

top10_plot = top10.sort_values("electricity_generation", ascending=True)

max_value = top10["electricity_generation"].max()
top_country = top10.iloc[0]["country"]

MAX_COLOR = "#ECFF16"       
NORMAL_COLOR = "#21A098"    
colors = [
    MAX_COLOR if country == top_country else NORMAL_COLOR
    for country in top10_plot["country"]
]


fig = go.Figure()

fig.add_trace(go.Bar(
    x=top10_plot["electricity_generation"],
    y=top10_plot["country"],
    orientation="h",
    marker=dict(
        color=colors,
        line=dict(color="black", width=0.6)
    ),
    text=[
        f"{value/1000:.0f}K" if value >= 1000 else f"{value:.0f}"
        for value in top10_plot["electricity_generation"]
    ],
    textposition="outside",
    textfont=dict(color="black", size=11),
    hovertemplate="<b>%{y}</b><br>Electricity Generation: %{x:,.0f} TWh<extra></extra>",
    showlegend=False
))

# Legend
fig.add_trace(go.Bar(
    x=[None],
    y=[None],
    name="Maximum Generation",
    marker=dict(color=MAX_COLOR, line=dict(color="black", width=0.4))
))

fig.add_trace(go.Bar(
    x=[None],
    y=[None],
    name="Standard Countries",
    marker=dict(color=NORMAL_COLOR, line=dict(color="black", width=0.4))
))


fig.update_layout(
    title=dict(
        text=f"Top 10 Countries by Electricity Generation ({latest_year})",
        x=0.5,
        font=dict(size=16, color="black", family="Arial Black")
    ),

    paper_bgcolor="white",
    plot_bgcolor="white",

    width=1250,
    height=600,

    margin=dict(l=300, r=90, t=90, b=80),

    font=dict(color="black", family="Arial"),

    legend=dict(
        x=0.985,
        y=0.97,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,1)",
        bordercolor="lightgray",
        borderwidth=1,
        font=dict(size=9, color="black"),
        itemwidth=30,
        traceorder="normal",
        title=None
    ),

    xaxis=dict(
        title=dict(
            text="Electricity Generation (TWh)",
            font=dict(size=15, color="black", family="Arial Black")
        ),

        range=[0, max_value * 1.33],

        zeroline=True,
        zerolinecolor="black",
        zerolinewidth=1,

        showgrid=True,
        gridcolor="lightgray",
        griddash="dot",

        tickfont=dict(size=10, color="black"),

        showline=True,
        linecolor="black",
        linewidth=1,
        mirror=True
    ),

    yaxis=dict(
        title=dict(
            text="Country Name",
            font=dict(size=15, color="black", family="Arial Black")
        ),

        tickfont=dict(size=10, color="black"),

        showgrid=False,

        showline=True,
        linecolor="black",
        linewidth=1,
        mirror=True
    ),

    bargap=0.22
)

fig.show()